# Transformer 작동방식

입력 : 토큰들의 시퀀스(sequence)
출력 : 딱 하나의 결과물이지만, 그 하나가 확률 분포(probability distribution) 형태입니다. 즉, "다음에 올 수 있는 토큰 각각"에 대한 확률값들의 집합입니다.

그래서 "출력이 하나다"라고 해도, 실제로는 그 안에 수 많은 숫자(확률값)가 들어있는 겁니다. 예를 들어 어휘 사전에 128000개의 토큰이 있다면, 모델은 128000개의 확률값을 출력합니다. "다음에 올 토큰이 이것일 확률", "저것일 확률" 이런 식으로 가능한 모든 토큰 각각에 대해 하나씩요.

1~2주차에 설명했던 내용을 떠올려 보시면, Transformer의 동작 흐름은 다음과 같습니다:

입력 시퀀스를 모델에 넣는다
모델이 "다음 토큰 후보들"에 대한 확률 분포를 계산해서 출력한다
이 확률들 중에서 하나를 **선택(샘플링)**해서, 그것을 실제 다음 토큰으로 결정한다

그리고 가장 일반적인 방식은, 가장 높은 확률을 받은 토큰을 그대로 다음 토큰으로 선택하는 것입니다. (이를 "greedy decoding"이라고 부릅니다.)

# 1. 토큰을 하나씩 생성하는 과정 (Autoregressive Generation)

방금 배운 것처럼, 모델은 입력 시퀀스를 받아서 "다음 토큰 확률 분포"를 하나 출력합니다. 그런데 실제 텍스트 생성은 이 과정을 반복해서 이루어집니다:

입력 시퀀스를 모델에 넣는다
다음 토큰의 확률 분포를 받는다
그중 가장 확률 높은 토큰을 고른다
그 토큰을 입력 시퀀스 끝에 이어붙여서 다시 모델에 넣는다
2~4번을 반복한다

이렇게 한 번에 토큰 하나씩 뱉어내기 때문에, ChatGPT 같은 서비스에서 답변이 타자기처럼 한 글자(토큰)씩 스트리밍되어 나오는 겁니다. 그리고 "다음 토큰"을 고를 때, 이전에 이미 생성된 토큰들과 일관성을 유지하려는 경향이 있기 때문에 결과적으로 훨씬 더 정확하고 자연스러운 답변이 나옵니다. (추론 모델이 "한 토큰씩 생각한다"는 표현도 이런 맥락입니다.)

# 2. 실제 예시: "파란색을 본 적 없는 사람에게 파란색을 설명해줘"

실제로 OpenAI API를 이용해 각 단계마다 가장 가능성 높은 토큰 몇 개와 그 확률을 직접 뽑아서 시각화 합니다. 

**"Explain blue to someone who has never seen it in one sentence"**라는 프롬프트에 대해:

첫 번째 토큰: "Blue"라는 단어가 확률 **99.9999%**로 거의 확실하게 선택됨 (두 번째 후보도 형태만 다른 "blue", 세 번째 후보 "Imagination"은 확률이 매우 낮음)
두 번째 토큰 ("Blue" 다음): "is"(62%), "feels"(38%) 같은 후보 중 선택
세 번째 토큰: "the", "a" 등
이런 식으로 이어지면서 "Blue is the cool, calming sensation..." 같은 표현력 있는 문장이 완성됨


# 3. 온도(Temperature)란?

```
response = openai.ChatCompletion.create(
    model=model,
    messages=messages,
    temperature=temperature
)
```
* temperture = 0 : 매번 확률이 가장 높은 토큰만 선택 -> 같은 입력에는 항상 같은 (또는 거의 같은) 결과가 나옴
* temperture 가 높을 수록 : 확률이 가장 높은 토큰이 아니어도, 확률에 따라 샘플링 (무작위 추출)을 하게 됩 -> 다양한 답변이 나올 가능성이 커짐

이를 "더 창의적이다"라고 표현하는건 건 정확하지 않고, "더 다양하다"라고 표현하는게 더 정확합니다. 즉 temperture는 "다음 토큰을 고르는 방식"을 조절하는 것으로 0이면 항상 1등만 뽑고, 높아질수록 1등이 아닌 토큰도 뽑힐 여지를 더 많이 준다는 뜻



In [4]:
!pip install visualizer


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\airtr\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [5]:
from visualizer import TokenPredictor, create_token_graph, visualize_predictions

message = "In one sentence, describe the color orange to someone who has never been able to see"
model_name = "gpt-4.1-mini"

predictor = TokenPredictor(model_name)
predictions = predictor.predict_tokens(message)
G = create_token_graph(model_name, predictions)
plt = visualize_predictions(G)
plt.show()

ModuleNotFoundError: No module named 'visualizer'

# 오디오 파일로부터 회의록 작성하기

In [1]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6
!pip install torch-directml

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 1.9.2 requires datasets>=4.7.0, but you have datasets 3.6.0 which is incompatible.


In [2]:
import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

In [3]:
import os
from dotenv import load_dotenv

hf_token = os.getenv('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)

HF key looks good so far


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:

LLAMA = "meta-llama/Llama-3.1-8B-Instruct"

In [5]:
audio_filename = "denver_extract.mp3"

# Open the file

audio_file = open(audio_filename, "rb")

# STEP 1: Transcribe Audio

In [ ]:
from transformers import pipeline
import torch_directml

device = torch_directml.device()

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    dtype=torch.float16,
    device=device,
    return_timestamps=True
)

result = pipe(audio_filename)
transcription = result["text"]
print(transcription)



config.json: 0.00B [00:00, ?B/s]

d:\anaconda3\envs\llm_new\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\airtr\.cache\huggingface\hub\models--openai--whisper-medium.en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use privateuseone:0
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


In [ ]:
open_source_transcription = transcription

# Option 2: Use OpenAI for Transcription

In [ ]:
AUDIO_MODEL = "gpt-4o-mini-transcribe"

openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)
transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")
print(transcription)

In [ ]:
display(Markdown(open_source_transcription))
print("\n\n")
display(Markdown(transcription))

# STEP 2: Analyze & Report

In [ ]:
system_message = """
당신은 회의 녹취록을 바탕으로 회의록을 작성합니다. 요약, 주요 논의 사항,
핵심 결론, 담당자가 명시된 실행 항목을 포함하여, 코드 블록 없이 마크다운 형식으로 작성합니다.
"""

user_prompt = f"""
아래는 덴버 시의회 회의의 발췌 녹취록입니다.
다음 내용을 포함하여 코드 블록 없이 마크다운 형식으로 회의록을 작성해 주세요:
- 참석자, 장소, 날짜가 포함된 요약
- 논의 사항
- 핵심 결론
- 담당자가 명시된 실행 항목

녹취록:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)